## Loading the packages and pre-build models and making the input data ready

In [1]:
## Importing the necessary packages
import pandas as pd
import numpy as np
import joblib
import shap

In [2]:
## Loading all the models
logistic_regression_model = joblib.load("../../../models/heart/new_models/new_logistic_regression_model.pkl")
random_forest_model = joblib.load("../../../models/heart/new_models/new_random_forest_model.pkl")
encoder = joblib.load("../../../models/heart/new_models/new_one_hot_encoder.pkl")
scaler = joblib.load("../../../models/heart/new_models/new_standard_scaler.pkl")

In [3]:
import pandas as pd
import joblib

input_data = pd.DataFrame([{
    "age": 63,
    "sex": "Male",
    "cp": "Typical Angina",
    "trestbps": 145,
    "chol": 233,
    "fbs": "Yes",
    "restecg": "Left Ventricular Hypertrophy",
    "thalach": 150,
    "exang": "No",
    "oldpeak": 2.3,
    "slope": "Downsloping",
    "ca": 0,
    "thal": "Fixed Defect"
}])

original_input_data = input_data.copy()

binary_map = {"Yes":1 , "No":0}
gender_map = {"Male":1,"Female":0}
chest_pain_map = {"Typical Angina":1 , "Atypical Angina": 2 , "Non-anginal Pain": 3 , "Asymptomatic": 4}
restecg_map = {"Normal": 0 , "ST-T Wave Abnormality": 1 , "Left Ventricular Hypertrophy": 2}
slope_map = {"Upsloping": 1 , "Flat": 2 , "Downsloping": 3}
thal_map = {"Normal": 3 , "Fixed Defect": 6 , "Reversible Defect": 7}

num_cols = ['age','trestbps','chol','thalach','oldpeak','ca']
binary_cols = ['fbs','exang']
categorical_cols = ['cp','restecg','slope','thal']

input_data["sex"] = input_data["sex"].map(gender_map)
for col in binary_cols:
    input_data[col] = input_data[col].map(binary_map)
    
input_data["cp"] = input_data["cp"].map(chest_pain_map)
input_data["restecg"] = input_data["restecg"].map(restecg_map)
input_data["slope"] = input_data["slope"].map(slope_map)
input_data["thal"] = input_data["thal"].map(thal_map)

encoder = joblib.load("../../../models/heart/new_models/new_one_hot_encoder.pkl")
encoded_array = encoder.transform(input_data[categorical_cols])
encoded_df = pd.DataFrame(encoded_array,columns=encoder.get_feature_names_out(categorical_cols))
input_data = input_data.drop(columns=categorical_cols)
input_data = pd.concat([input_data,encoded_df], axis=1)
scaler = joblib.load("../../../models/heart/new_models/new_standard_scaler.pkl")
input_data[num_cols] = scaler.transform(input_data[num_cols])

input_data.head()

,age,sex,trestbps,chol,fbs,thalach,exang,oldpeak,ca,cp_2.0,cp_3.0,cp_4.0,restecg_1.0,restecg_2.0,slope_2.0,slope_3.0,thal_6.0,thal_7.0
0,0.935799,1,0.708191,-0.295626,1,0.065524,0,1.112556,-0.720577,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0


## For the explainability of the logistic regression model

In [4]:
lr_prediction = logistic_regression_model.predict(input_data)[0]
lr_probability = logistic_regression_model.predict_proba(input_data)[0][1]

# =====================================================
# CHECK MODEL COEFFICIENTS
# =====================================================

feature_names = input_data.columns
coefficients = logistic_regression_model.coef_[0]
coef_df = pd.DataFrame({"Feature": feature_names,"Coefficient": coefficients})

print(coef_df.sort_values(by="Coefficient",ascending=False))

# =====================================================
# PREDICTION RESULT
# =====================================================

lr_result = ("YES" if lr_prediction == 1 else "NO")

print(f"\nHeart Disease : {lr_result}\n")
print(f"Risk Probability : {lr_probability:.2f}\n")

# =====================================================
# RECOMMENDATION
# =====================================================

if lr_probability >= 0.70:
    print("Recommendation : Immediate doctor consultation is advised.")
elif lr_probability >= 0.40:
    print("Recommendation : Regular doctor consultation is advised.")
else:
    print("Recommendation : No immediate consultation needed.")

# =====================================================
# SHAP EXPLAINER
# =====================================================

X_train = pd.read_csv("../../../data/heart/processed/new_dataset_cleveland/X_train.csv")

lr_explainer = shap.Explainer(logistic_regression_model,X_train)

# =====================================================
# SHAP VALUES
# =====================================================

lr_shap_values = lr_explainer(input_data)

print("\nExplanation : \n")

for i, col in enumerate(input_data.columns):
    impact = lr_shap_values.values[0][i]

    # =================================================
    # ORIGINAL COLUMNS
    # =================================================

    if col in original_input_data.columns:
        value = original_input_data.iloc[0][col]
        if impact > 0:
            print(f"{col} : {value} , Increased risk of heart disease.")
        elif impact < 0:
            print(f"{col} : {value} , Decreased risk of heart disease.")

    # =================================================
    # ENCODED COLUMNS
    # =================================================

    else:
        split_col = col.split("_")
        original_feature = split_col[0]
        encoded_value = "_".join(split_col[1:])
        if input_data.iloc[0][col] == 1:
            if original_feature == "cp":
                cp_reverse_map = {"1": "Typical Angina","2": "Atypical Angina","3": "Non-anginal Pain","4": "Asymptomatic"}
                encoded_value = cp_reverse_map.get(encoded_value,encoded_value)
            elif original_feature == "restecg":
                restecg_reverse_map = {"0": "Normal","1": "ST-T Wave Abnormality","2": "Left Ventricular Hypertrophy"}
                encoded_value = restecg_reverse_map.get(encoded_value,encoded_value)
            elif original_feature == "slope":
                slope_reverse_map = {"1": "Upsloping","2": "Flat","3": "Downsloping"}
                encoded_value = slope_reverse_map.get(encoded_value,encoded_value)
            elif original_feature == "thal":
                thal_reverse_map = {"3": "Normal","6": "Fixed Defect","7": "Reversible Defect"}
                encoded_value = thal_reverse_map.get(encoded_value,encoded_value)

            # =========================================
            # FINAL OUTPUT
            # =========================================

            if impact > 0:
                print(f"{original_feature} : {encoded_value} , Increased risk of heart disease.")
            elif impact < 0:
                print(f"{original_feature} : {encoded_value} , Decreased risk of heart disease.")

        Feature  Coefficient
8            ca     0.709449
17     thal_7.0     0.654769
1           sex     0.539051
11       cp_4.0     0.535286
6         exang     0.423469
7       oldpeak     0.401466
14    slope_2.0     0.284935
2      trestbps     0.262496
13  restecg_2.0     0.161489
3          chol     0.141052
9        cp_2.0     0.026688
15    slope_3.0    -0.001697
12  restecg_1.0    -0.003274
0           age    -0.024057
16     thal_6.0    -0.030573
4           fbs    -0.210653
10       cp_3.0    -0.365855
5       thalach    -0.386379

Heart Disease : NO

Risk Probability : 0.36

Recommendation : No immediate consultation needed.

Explanation : 

age : 63 , Decreased risk of heart disease.
sex : Male , Increased risk of heart disease.
trestbps : 145 , Increased risk of heart disease.
chol : 233 , Decreased risk of heart disease.
fbs : Yes , Decreased risk of heart disease.
thalach : 150 , Decreased risk of heart disease.
exang : No , Decreased risk of heart disease.
oldpeak :

In [5]:
rf_prediction = random_forest_model.predict(input_data)[0]
rf_probabilty = random_forest_model.predict_proba(input_data)[0][1]
rf_result = "YES" if rf_prediction==1 else "NO"
print(f"Heart Disease : {rf_result}\n")
print(f"Risk Probability : {rf_probabilty:.2f}\n")
if rf_probabilty>=0.70:
    print(f"Recommendation : Immediate doctor consultation is advised.")
elif rf_probabilty>=0.40:
    print(f"Recommendation : Regular doctor consultation is advised.")
else:
    print(f"Recommendation : No immediate consultation needed.")

## Building the explainer model for the logistic regression
rf_explainer = shap.TreeExplainer(random_forest_model)

## For the particular input - output the explanation
rf_shap_values = rf_explainer.shap_values(input_data)
rf_values = rf_shap_values[:,:,1]
print("\nExplanation : \n")
for i,col in enumerate(input_data.columns):
    impact = rf_values[0][i]
    if col in original_input_data.columns:
        value = original_input_data.iloc[0][col]
        if impact>0:
            print(f"{col} : {value} , Increased risk of heart disease.")
        elif impact<0:
            print(f"{col} : {value} , Decreased risk of heart disease.")
    else:
        split_col = col.split("_")
        original_feature = split_col[0]
        encoded_value = "_".join(split_col[1:])
        if input_data.iloc[0][col] == 1:
            if original_feature == "cp":
                cp_reverse_map = {"1": "Typical Angina","2": "Atypical Angina","3": "Non-anginal Pain","4": "Asymptomatic"}
                encoded_value = cp_reverse_map.get(encoded_value,encoded_value)
            elif original_feature == "restecg":
                restecg_reverse_map = {"0": "Normal","1": "ST-T Wave Abnormality","2": "Left Ventricular Hypertrophy"}
                encoded_value = restecg_reverse_map.get(encoded_value,encoded_value)
            elif original_feature == "slope":
                slope_reverse_map = {"1": "Upsloping","2": "Flat","3": "Downsloping"}
                encoded_value = slope_reverse_map.get(encoded_value,encoded_value)
            elif original_feature == "thal":
                thal_reverse_map = {"3": "Normal","6": "Fixed Defect","7": "Reversible Defect"}
                encoded_value = thal_reverse_map.get(encoded_value,encoded_value)
            if impact > 0:
                print(f"{original_feature} : {encoded_value} , Increased risk of heart disease.")
            elif impact < 0:
                print(f"{original_feature} : {encoded_value} , Decreased risk of heart disease.")

Heart Disease : NO

Risk Probability : 0.12

Recommendation : No immediate consultation needed.

Explanation : 

age : 63 , Increased risk of heart disease.
sex : Male , Increased risk of heart disease.
trestbps : 145 , Decreased risk of heart disease.
chol : 233 , Decreased risk of heart disease.
fbs : Yes , Decreased risk of heart disease.
thalach : 150 , Decreased risk of heart disease.
exang : No , Decreased risk of heart disease.
oldpeak : 2.3 , Increased risk of heart disease.
ca : 0 , Decreased risk of heart disease.
restecg : 2.0 , Decreased risk of heart disease.
slope : 3.0 , Decreased risk of heart disease.
thal : 6.0 , Increased risk of heart disease.


## Saving the explainer models

In [6]:
joblib.dump(lr_explainer,"../../../models/heart/new_models/new_lr_shap_explainer.pkl")
joblib.dump(rf_explainer,"../../../models/heart/new_models/new_rf_shap_explainer.pkl")

['../../../models/heart/new_models/new_rf_shap_explainer.pkl']

In [7]:
print(input_data.columns.tolist())
print(X_train.columns.tolist())

['age', 'sex', 'trestbps', 'chol', 'fbs', 'thalach', 'exang', 'oldpeak', 'ca', 'cp_2.0', 'cp_3.0', 'cp_4.0', 'restecg_1.0', 'restecg_2.0', 'slope_2.0', 'slope_3.0', 'thal_6.0', 'thal_7.0']
['age', 'sex', 'trestbps', 'chol', 'fbs', 'thalach', 'exang', 'oldpeak', 'ca', 'cp_2.0', 'cp_3.0', 'cp_4.0', 'restecg_1.0', 'restecg_2.0', 'slope_2.0', 'slope_3.0', 'thal_6.0', 'thal_7.0']
